# Day 3 — Grounded Generation & Citation
### AI Clinical Decision Support Lite Hackathon · Plan B

**Prepared by the Day 3 Notebook Council** (see `notebooks/COUNCIL.md` for reviewer credits)

Day 2 proved your retrieval is trustworthy. Today you constrain the model so tightly that
every word it generates can be traced back to a real page in a real guideline — including
knowing when to say "I don't know" instead of guessing.

**By the end of this notebook you will be able to:**
1. Write a system prompt that structurally forbids answering from outside knowledge
2. Validate a generated answer against `schema/response_schema.json`
3. Build and test a refusal case that triggers correctly on an out-of-scope question
4. Explain, with evidence, why exact wording matters more than paraphrasing here

> This notebook uses **Groq** (Qwen 3.6 27B) for fast, structured clinical generation
> with citation enforcement and schema validation.


## 0. Setup — Rebuild the Index from Day 1/2


In [1]:
import sys, os, re
sys.path.append(os.path.abspath(".."))

import json
import config
from ingest import load_pdfs, chunk_documents, build_index
from query import retrieve

pages = load_pdfs(config.DATA_DIR)
chunks = chunk_documents(pages)
vectordb = build_index(chunks)
print(f"\nIndex ready: {len(chunks)} chunks from {len(pages)} pages.")


Loading Guideline for the pharmacological treatment of hypertension in adults.pdf ...
  -> 61 pages loaded
Loading WHO_Hypertension_Guideline_2021.pdf ...
  -> 13 pages loaded
Embedding 217 chunks using 'local' provider ...
Done. Index saved to chroma_db/

Index ready: 217 chunks from 74 pages.


## 1. The Grounding System Prompt

A grounding prompt needs four parts: a **role** that isn't a general medical advisor, an
explicit **context boundary**, a required **output format**, and an **escape hatch** for
insufficient evidence. Here's a working version — read it fully before running it.


In [2]:
GROUNDING_SYSTEM_PROMPT = """You are a citation-bound clinical evidence assistant.

RULES — follow every one exactly:
1. Answer ONLY using the context passages provided below. Never use outside medical knowledge.
2. Every claim in your "recommendation" must be directly supported by the "evidence" you cite.
3. You MUST return your answer as JSON matching exactly this structure:
   {"recommendation": "...", "evidence": "...", "citations": [{"document": "...", "section": "...", "page": N}], "confidence": "high" | "medium" | "low" | "insufficient"}
4. If the context does not contain enough information to answer confidently, set confidence to "insufficient", leave evidence and citations empty, and write a plain refusal in "recommendation" instead of guessing.
5. Never invent a citation. Never soften a refusal into a partial guess.
6. Return ONLY the JSON object. No thinking tags, no markdown, no explanations.
"""

print(GROUNDING_SYSTEM_PROMPT)


You are a citation-bound clinical evidence assistant.

RULES — follow every one exactly:
1. Answer ONLY using the context passages provided below. Never use outside medical knowledge.
2. Every claim in your "recommendation" must be directly supported by the "evidence" you cite.
3. You MUST return your answer as JSON matching exactly this structure:
   {"recommendation": "...", "evidence": "...", "citations": [{"document": "...", "section": "...", "page": N}], "confidence": "high" | "medium" | "low" | "insufficient"}
4. If the context does not contain enough information to answer confidently, set confidence to "insufficient", leave evidence and citations empty, and write a plain refusal in "recommendation" instead of guessing.
5. Never invent a citation. Never soften a refusal into a partial guess.
6. Return ONLY the JSON object. No thinking tags, no markdown, no explanations.



### Checkpoint 1

Read rule 5 again: *"Never invent a citation."* This is the single most common failure
mode in ungrounded RAG systems — a model that sounds confident and cites a page number
that, when you check it, doesn't actually say what the model claims. Every citation your
system produces this week should be one you could click through and verify by hand.


## 2. Validate the Response Schema

`schema/response_schema.json` — already in your starter kit — is a real JSON Schema that
enforces the shape above, *and* enforces rule 4 structurally: if `confidence` isn't
`"insufficient"`, the schema requires non-empty `evidence` and at least one citation.

Let's load it and test it against a valid answer and a deliberately broken one.


In [3]:
from jsonschema import validate, ValidationError

with open("../schema/response_schema.json") as f:
    schema = json.load(f)

good_answer = {
    "recommendation": "Start with a thiazide-type diuretic, an ACE inhibitor/ARB, or a long-acting calcium channel blocker.",
    "evidence": "WHO recommends the use of drugs from any of the following three classes... as an initial treatment.",
    "citations": [{"document": "WHO_Hypertension_Guideline_2021", "section": "3.4 Drug classes", "page": 8}],
    "confidence": "high",
}

broken_answer = {
    "recommendation": "Take 10mg of drug X daily.",
    "evidence": "",
    "citations": [],
    "confidence": "high",   # high confidence but no evidence — should be rejected
}

for label, answer in [("Well-formed answer", good_answer), ("High confidence, no evidence", broken_answer)]:
    try:
        validate(instance=answer, schema=schema)
        print(f"{label}: PASSED validation")
    except ValidationError as e:
        print(f"{label}: REJECTED — {e.message}")


Well-formed answer: PASSED validation
High confidence, no evidence: REJECTED — '' should be non-empty


### Checkpoint 2

The second case should be **rejected**. If it passed instead, your schema (or your
understanding of it) has a gap — a "high confidence" answer with zero supporting evidence
is exactly the hallucination pattern grounding is supposed to prevent.


## 3. Build the Generation Function

This function does the real work: retrieve context, assemble the grounded prompt, and
call the model via **Groq** (Qwen 3.6 27B). The response is parsed to strip any
`<think>` tags, then validated against the JSON schema.


In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model=config.GROQ_MODEL,
    api_key=config.GROQ_API_KEY,
    temperature=0,
)


def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(
        f"[{doc.metadata.get('document_name')}, "
        f"Page {doc.metadata.get('page_number')}]\n{doc.page_content}"
        for doc, _ in retrieved_chunks
    )
    return f"""{GROUNDING_SYSTEM_PROMPT}

Context:
{context}

Question: {question}

Return ONLY a single JSON object with keys: recommendation, evidence, citations, confidence. Nothing else."""


def parse_llm_json(raw):
    cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', cleaned, re.DOTALL)
    if m:
        return json.loads(m.group(1))
    m = re.search(r'\{[^{}]*"recommendation"[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', cleaned, re.DOTALL)
    if m:
        return json.loads(m.group())
    raise ValueError(f"Could not parse JSON from: {cleaned[:200]}")


def generate_grounded_answer(question, k=3):
    results = retrieve(vectordb, question, k=k)
    prompt = build_prompt(question, results)
    response = llm.invoke(prompt)
    answer = parse_llm_json(response.content)
    return answer, prompt, results


In [5]:
answer, prompt_used, _ = generate_grounded_answer(
    "What is the target blood pressure for a patient with cardiovascular disease?"
)

print("\n--- Generated answer ---")
print(json.dumps(answer, indent=2))

print("\n--- Schema validation ---")
try:
    validate(instance=answer, schema=schema)
    print("PASSED")
except ValidationError as e:
    print("REJECTED:", e.message)



--- Generated answer ---
{
  "recommendation": "The target systolic blood pressure treatment goal is <130 mmHg.",
  "evidence": "WHO recommends a target systolic blood pressure treatment goal of <130 mmHg in patients with hypertension and known cardiovascular disease (CVD).",
  "citations": [
    {
      "document": "Guideline for the pharmacological treatment of hypertension in adults",
      "section": "3.6 Target blood pressure",
      "page": 28
    }
  ],
  "confidence": "high"
}

--- Schema validation ---
PASSED


## 4. Build and Test a Refusal Case

Your live demo on Day 5 must include at least one refusal that works on command. Let's
build one now, using a question this source genuinely cannot answer.


In [6]:
def generate_with_refusal_check(question, confidence_threshold=0.3):
    results = retrieve(vectordb, question, k=3)
    top_score = results[0][1] if results else -999

    if top_score < confidence_threshold:
        return {
            "recommendation": (
                "I could not find enough information in the indexed guideline to answer "
                "this confidently. This source does not appear to cover this topic \u2014 try "
                "rephrasing, or consult a clinician directly."
            ),
            "evidence": "",
            "citations": [],
            "confidence": "insufficient",
        }
    answer, _, _ = generate_grounded_answer(question)
    return answer


out_of_scope_question = "What screening interval does this guideline recommend for breast cancer?"
refusal_answer = generate_with_refusal_check(out_of_scope_question)

print(json.dumps(refusal_answer, indent=2))
print("\n--- Schema validation ---")
validate(instance=refusal_answer, schema=schema)
print("PASSED — refusal is schema-valid")


{
  "recommendation": "I could not find enough information in the indexed guideline to answer this confidently. This source does not appear to cover this topic \u2014 try rephrasing, or consult a clinician directly.",
  "evidence": "",
  "citations": [],
  "confidence": "insufficient"
}

--- Schema validation ---
PASSED — refusal is schema-valid


### Checkpoint 3

Note the `confidence_threshold` used above is illustrative — because embedding score
ranges differ by model, you'll calibrate the real number on Day 4 using your own
Precision@k data from Day 2. For today, the important thing is that the refusal path
**exists, triggers correctly, and produces schema-valid output** — not the exact
threshold value.

Save the exact question above (`{out_of_scope_question}`) — it's your rehearsed refusal
demo for Day 5.


## 5. Day 3 Self-Check

- [x] Your grounding prompt includes all 4 parts: role, context boundary, output format, escape hatch
- [x] A high-confidence answer with no evidence is correctly **rejected** by the schema
- [x] Your refusal case produces valid, schema-passing JSON — not a plain-text apology
- [x] You've saved the exact out-of-scope question you'll use in your Day 5 demo

## What's Next

Day 4's notebook takes the `confidence_threshold` you used loosely here and calibrates it
properly against real Precision@k data, then adds a second safety layer: catching claims
that slip past the prompt and aren't actually supported by the retrieved text.
